In [21]:
import ee
from tgbs_rs.config.config import GEE_PROJECT

In [22]:
from tgbs_rs.config.config import (
    BASELINE_START, 
    CURRENT_END, 
    S2_BASELINE_START, 
    S2_ALL_BANDS, 
    HLS_MERGED_BANDS, 
    DRIVE_FOLDER,
    AOI_PATHS
)
from tgbs_rs.data.sensors.hls.hls_preprocessing import get_hls_merged_collection
from tgbs_rs.data.sensors.sentinel.sentinel_preprocessing import get_s2_sr_collection
from tgbs_rs.utils import build_default_sites_featurecollection, load_site_feature
from tgbs_rs.metrics.temporal import build_period_composites, collection_to_site_timeseries
from tgbs_rs.io import export_table_to_drive

In [23]:
# Authenticate and Initialize Earth Engine
ee.Authenticate()
ee.Initialize(project= "charrell-personal")

In [24]:
# Build Site Feature Collection
sites_fc = build_default_sites_featurecollection()

# Create Feature for biodiversity corridor
corridor_feature = load_site_feature(
    AOI_PATHS["bio_corridor"],
    site_id="bio_corridor",
    site_name="Biodiversity Corridor",
    site_category="corridor",
)

sites_fc = sites_fc.merge(ee.FeatureCollection([corridor_feature]))

#### Build HLS Annual and Monthly Composite Collections

In [25]:
# Build HLS Source Collection
hls_col = get_hls_merged_collection(
    aoi=sites_fc,
    start_date=ee.Date(BASELINE_START),
    end_date=ee.Date(CURRENT_END),
    apply_water_masking=True
)

In [26]:
# Build HLS Annual Multiband Composite
hls_annual_col = build_period_composites(
    collection=hls_col,
    bands=HLS_MERGED_BANDS,
    start_date=ee.Date(BASELINE_START),
    end_date=ee.Date(CURRENT_END),
    temporal_scale="annual",
    composite_stat="median"
)

In [27]:
# Build HLS Monthly Multiband Composite
hls_monthly_col = build_period_composites(
    collection=hls_col,
    bands=HLS_MERGED_BANDS,
    start_date=ee.Date(BASELINE_START),
    end_date=ee.Date(CURRENT_END),
    temporal_scale="monthly",
    composite_stat="median"
)

In [28]:
# Reduce each annual image over all sites
hls_annual_site_fc = collection_to_site_timeseries(
    collection=hls_annual_col,
    sites_fc=sites_fc,
    bands=HLS_MERGED_BANDS,
    reducer=ee.Reducer.mean(),
    scale=30,
    tile_scale=4
)

In [29]:
# Reduce each annual image over all sites
hls_monthly_site_fc = collection_to_site_timeseries(
    collection=hls_monthly_col,
    sites_fc=sites_fc,
    bands=HLS_MERGED_BANDS,
    reducer=ee.Reducer.mean(),
    scale=30,
    tile_scale=4
)

In [30]:
# Export Long ee.FeatureCollection to Drive as CSV
export_table_to_drive(
    collection=hls_annual_site_fc,
    description="tgbs_hls_annual_composite_all_bands_2014_2025_full_w_corridor",
    folder=DRIVE_FOLDER,
    fileNamePrefix="tgbs_hls_annual_composite_all_bands_2014_2025_full_w_corridor",
    fileFormat="CSV"
)

Export started: {'state': 'READY', 'description': 'tgbs_hls_annual_composite_all_bands_2014_2025_full_w_corridor', 'priority': 100, 'creation_timestamp_ms': 1776953289061, 'update_timestamp_ms': 1776953289061, 'start_timestamp_ms': 0, 'task_type': 'EXPORT_FEATURES', 'id': 'NDQ6DQ63Y6LH25BYUQZ47H5Y', 'name': 'projects/charrell-personal/operations/NDQ6DQ63Y6LH25BYUQZ47H5Y'}


In [31]:
# Export Long ee.FeatureCollection to Drive as CSV
export_table_to_drive(
    collection=hls_monthly_site_fc,
    description="tgbs_hls_monthly_composite_all_bands_2014_2025_full_w_corridor",
    folder=DRIVE_FOLDER,
    fileNamePrefix="tgbs_hls_monthly_composite_all_bands_2014_2025_full_w_corridor",
    fileFormat="CSV"
)

Export started: {'state': 'READY', 'description': 'tgbs_hls_monthly_composite_all_bands_2014_2025_full_w_corridor', 'priority': 100, 'creation_timestamp_ms': 1776953291588, 'update_timestamp_ms': 1776953291588, 'start_timestamp_ms': 0, 'task_type': 'EXPORT_FEATURES', 'id': 'SU3FV35QYHIEKCL7TSN7AENP', 'name': 'projects/charrell-personal/operations/SU3FV35QYHIEKCL7TSN7AENP'}


#### Build Sentinel-2 Annual and Monthly Composite Collections

In [32]:
# Build Sentinel-2 SR Source Collection
s2_col = get_s2_sr_collection(
    aoi=sites_fc,
    start_date=ee.Date(S2_BASELINE_START),
    end_date=ee.Date(CURRENT_END),
    apply_water_masking=True
)

In [33]:
# Build HLS Annual Multiband Composite
s2_annual_col = build_period_composites(
    collection=s2_col,
    bands=S2_ALL_BANDS,
    start_date=ee.Date(S2_BASELINE_START),
    end_date=ee.Date(CURRENT_END),
    temporal_scale="annual",
    composite_stat="median"
)

In [34]:
# Build HLS Monthly Multiband Composite
s2_monthly_col = build_period_composites(
    collection=s2_col,
    bands=S2_ALL_BANDS,
    start_date=ee.Date(S2_BASELINE_START),
    end_date=ee.Date(CURRENT_END),
    temporal_scale="monthly",
    composite_stat="median"
)

In [35]:
# Reduce each annual image over all sites
s2_annual_site_fc = collection_to_site_timeseries(
    collection=s2_annual_col,
    sites_fc=sites_fc,
    bands=S2_ALL_BANDS,
    reducer=ee.Reducer.mean(),
    scale=30,
    tile_scale=4
)

In [36]:
# Reduce each annual image over all sites
s2_monthly_site_fc = collection_to_site_timeseries(
    collection=s2_monthly_col,
    sites_fc=sites_fc,
    bands=S2_ALL_BANDS,
    reducer=ee.Reducer.mean(),
    scale=30,
    tile_scale=4
)

In [37]:
# Export Long ee.FeatureCollection to Drive as CSV
export_table_to_drive(
    collection=s2_annual_site_fc,
    description="tgbs_s2_annual_composite_all_bands_2019_2025_full_w_corridor",
    folder=DRIVE_FOLDER,
    fileNamePrefix="tgbs_s2_annual_composite_all_bands_2019_2025_full_w_corridor",
    fileFormat="CSV"
)

Export started: {'state': 'READY', 'description': 'tgbs_s2_annual_composite_all_bands_2019_2025_full_w_corridor', 'priority': 100, 'creation_timestamp_ms': 1776953315925, 'update_timestamp_ms': 1776953315925, 'start_timestamp_ms': 0, 'task_type': 'EXPORT_FEATURES', 'id': 'R6JCZHAA2SSPK2ZPPGXFIB4Q', 'name': 'projects/charrell-personal/operations/R6JCZHAA2SSPK2ZPPGXFIB4Q'}


In [38]:
# Export Long ee.FeatureCollection to Drive as CSV
export_table_to_drive(
    collection=s2_monthly_site_fc,
    description="tgbs_s2_monthly_composite_all_bands_2019_2025_full_w_corridor",
    folder=DRIVE_FOLDER,
    fileNamePrefix="tgbs_s2_monthly_composite_all_bands_2019_2025_full_w_corridor",
    fileFormat="CSV"
)

Export started: {'state': 'READY', 'description': 'tgbs_s2_monthly_composite_all_bands_2019_2025_full_w_corridor', 'priority': 100, 'creation_timestamp_ms': 1776953318386, 'update_timestamp_ms': 1776953318386, 'start_timestamp_ms': 0, 'task_type': 'EXPORT_FEATURES', 'id': 'QXDZJTC5B2E5QLAAHCBQ67MM', 'name': 'projects/charrell-personal/operations/QXDZJTC5B2E5QLAAHCBQ67MM'}
